In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import eelbrain
from eelbrain import *

## User settings

In [ ]:
MAINDIR = Path(r"C:/projects/emo_EEG")
LOC_PATH = MAINDIR / "data" / "biosemi_32ch_2mastoid_locs.csv"
TRF_DIR = MAINDIR / "data_pipeline" / "mTRF" / "eelbrain"/ "data"/ "trf"

models = {
    'envelope': ['envelope'],
    'envelope+onset': ['envelope', 'acoustic_onset'],
    'gammatone_8 + gammatone_on_8': ['gammatone_8', 'gammatone_on_8']
}
emotions = ['hap', 'sad'] 
SUBS = [f"sub-pilot_{i}" for i in range(1, 6)]

prp_sensors = ['AF3', 'AF4', 'Fz', 'F3', 'F4', 'FC1', 'FC2']

## Model 1: Broadband envelope

In [ ]:
# load the envelope-only model
rows = []
for subject in SUBS:
    for emotion in emotions:
        trf = eelbrain.load.unpickle(TRF_DIR / f'{subject}_{emotion}_envelope.pickle')
        rows.append([subject, emotion, trf.h, trf.proportion_explained])

data_envelope = eelbrain.Dataset.from_caselist(
    ['subject', 'emotion', 'trf', 'det'],
    rows
)

data_envelope['subject'] = Factor(data_envelope['subject'], random=True)
data_envelope['emotion'] = Factor(data_envelope['emotion'])

hap_ds = data_envelope.sub("emotion == 'hap'")
sad_ds = data_envelope.sub("emotion == 'sad'")

In [ ]:
test_envelope = eelbrain.testnd.TTestOneSample('det', data = hap_ds, tail = 1, pmin = 0.05)
p = eelbrain.plot.Topomap(test_envelope, clip = 'circle', w =2, vmin = -0.001, vmax =0.001)
cb = p.plot_colorbar(width=0.1)

In [ ]:
test_envelope = eelbrain.testnd.TTestOneSample('det', data = sad_ds, tail = 1, pmin = 0.05)
p = eelbrain.plot.Topomap(test_envelope, clip = 'circle', w =2, vmin = -0.001, vmax =0.001)
print(p.get_vlim())

In [ ]:
t_env = [0.040, 0.090, 0.140, 0.250, 0.400]

test_env_hap_vs_sad = eelbrain.testnd.TTestRelated(
    hap_ds['trf'],
    sad_ds['trf'],
    pmin=0.05
)

p = eelbrain.plot.TopoArray(
    test_env_hap_vs_sad,
    t=t_env,
    clip='circle',
    cmap='xpolar',
    axtitle=['hap', 'sad', 'hap vs sad']
)
cb = p.plot_colorbar(width=0.1)

In [ ]:
print(test_env_hap_vs_sad)
print(test_env_hap_vs_sad.c0_mean)
print(test_env_hap_vs_sad.c1_mean)

## Model 2: Broadband Envelope + Acoustic onset

In [ ]:
# load the envelope and envelope + onset model
rows = []
x_names = None

for subject in SUBS:
    for emotion in emotions:
        trf = eelbrain.load.unpickle(TRF_DIR / f'{subject}_{emotion}_envelope+onset.pickle')
        rows.append([subject, emotion, *trf.h, trf.proportion_explained])
        x_names = trf.x   # should be something like ('envelope', 'onset')

data_onset = eelbrain.Dataset.from_caselist(
    ['subject', 'emotion', *x_names, 'det'],
    rows
)

data_onset['subject'] = Factor(data_onset['subject'], random=True)
data_onset['emotion'] = Factor(data_onset['emotion'])

hap_on_ds = data_onset.sub("emotion == 'hap'")
sad_on_ds = data_onset.sub("emotion == 'sad'")

In [ ]:
test_onset = eelbrain.testnd.TTestOneSample('det', data = hap_on_ds, tail = 1, pmin = 0.05)
p = eelbrain.plot.Topomap(test_onset, clip = 'circle', w =2, vmin = -0.001, vmax =0.001)
print(p.get_vlim())

In [ ]:
test_onset = eelbrain.testnd.TTestOneSample('det', data = sad_on_ds, tail = 1, pmin = 0.05)
p = eelbrain.plot.Topomap(test_onset, clip = 'circle', w =2, vmin = -0.001, vmax =0.001)
print(p.get_vlim())

In [ ]:
t_env = [0.040, 0.090, 0.140, 0.250, 0.400]

test_env_hap_vs_sad = eelbrain.testnd.TTestRelated(
    hap_ds['Fem_ADS_sad_cont_1_envelope'],
    sad_ds['Fem_ADS_sad_cont_1_envelope'],
    pmin=0.05
)

p = eelbrain.plot.TopoArray(
    test_env_hap_vs_sad,
    t=t_env,
    clip='circle',
    cmap='xpolar',
    axtitle=['hap', 'sad', 'hap vs sad']
)
cb = p.plot_colorbar(width=0.1)

In [ ]:
t_env = [0.040, 0.090, 0.140, 0.250, 0.400]

hap_ds = data_onset.sub("emotion == 'hap'")
sad_ds = data_onset.sub("emotion == 'sad'")

test_env_hap_vs_sad = eelbrain.testnd.TTestRelated(
    hap_ds['Fem_ADS_sad_cont_1_onset'],
    sad_ds['Fem_ADS_sad_cont_1_onset'],
    pmin=0.05
)

p = eelbrain.plot.TopoArray(
    test_env_hap_vs_sad,
    t=t_env,
    clip='circle',
    cmap='xpolar',
    axtitle=['hap', 'sad', 'hap vs sad']
)
cb = p.plot_colorbar(width=0.1)

## Model 3: Full acoustic model

In [ ]:
# load the model
rows = []
x_names = None

for subject in SUBS:
    for emotion in emotions:
        trf = eelbrain.load.unpickle(TRF_DIR / f'{subject}_{emotion}_gammatone_8+gammatone_on_8.pickle')
        rows.append([subject, emotion, *trf.h, trf.proportion_explained])
        x_names = trf.x   # should be something like ('envelope', 'onset')

data_acoustic = eelbrain.Dataset.from_caselist(
    ['subject', 'emotion', *x_names, 'det'],
    rows
)

data_acoustic['subject'] = Factor(data_acoustic['subject'], random=True)
data_acoustic['emotion'] = Factor(data_acoustic['emotion'])

hap_acoustic_ds = data_acoustic.sub("emotion == 'hap'")
sad_acoustic_ds = data_acoustic.sub("emotion == 'sad'")

In [ ]:
res = eelbrain.testnd.TTestRelated(
    'det',
    'emotion',
    'hap',
    'sad',
    match='subject',
    data=data_acoustic,
    pmin=0.05,
)
print(res.p)

In [ ]:
p = eelbrain.plot.Topomap(res, clip='circle')

In [ ]:
hap_test_acoustic = eelbrain.testnd.TTestOneSample('det', data = hap_acoustic_ds, tail = 1, pmin = 0.05)
p = eelbrain.plot.Topomap(hap_test_acoustic, clip = 'circle', w =2, vmin = - 0.003, vmax = 0.003)
p.mark_sensors(prp_sensors, s=2, c='green')
cb = p.plot_colorbar(width=0.1)
print(p.get_vlim())

In [ ]:
sad_test_acoustic = eelbrain.testnd.TTestOneSample('det', data = sad_acoustic_ds, tail = 1, pmin = 0.05)
p = eelbrain.plot.Topomap(sad_test_acoustic, clip = 'circle', w =2, vmin = - 0.003, vmax = 0.003)
p.mark_sensors(prp_sensors, s=2, c='green')
print(p.get_vlim())

In [ ]:
trf_spectrogram = eelbrain.testnd.TTestOneSample("Fem_ADS_sad_cont_1_gammatone_8.sum('frequency')", data=data_acoustic, pmin=0.05)
trf_onset_spectrogram = eelbrain.testnd.TTestOneSample("Fem_ADS_sad_cont_1_gammatone_on_8.sum('frequency')", data=data_acoustic, pmin=0.05)

p = eelbrain.plot.TopoArray(
    [trf_spectrogram, trf_onset_spectrogram], 
    t=[0.050, 0.100, 0.150, 0.450], 
    axtitle=['gammatone_8', 'gammatone_on_8']
    )

In [ ]:
hap_strf_spectrogram = hap_acoustic_ds['Fem_ADS_sad_cont_1_gammatone_8'].mean(sensor=prp_sensors).smooth('frequency', window_samples=7, fix_edges=True)
hap_strf_onset_spectrogram = hap_acoustic_ds['Fem_ADS_sad_cont_1_gammatone_on_8'].mean(sensor=prp_sensors)
p = eelbrain.plot.Array(
    [hap_strf_spectrogram, hap_strf_onset_spectrogram],
    cmap='xpolar',
    axtitle=['gammatone_8', 'gammatone_on_8']
    )
cb = p.plot_colorbar(width = 0.1)

In [ ]:
sad_strf_spectrogram = sad_acoustic_ds['Fem_ADS_sad_cont_1_gammatone_8'].mean(sensor=prp_sensors).smooth('frequency', window_samples=7, fix_edges=True)
sad_strf_onset_spectrogram = sad_acoustic_ds['Fem_ADS_sad_cont_1_gammatone_on_8'].mean(sensor=prp_sensors)
p = eelbrain.plot.Array(
    [sad_strf_spectrogram, sad_strf_onset_spectrogram],
    cmap='xpolar',
    axtitle=['gammatone_8', 'gammatone_on_8']
    )
cb = p.plot_colorbar(width = 0.1)

In [ ]:
t_env = [0.040, 0.090, 0.140, 0.250, 0.400]

hap_trf_spectrogram = eelbrain.testnd.TTestOneSample()

## Model 4: Phoneme onset 

In [ ]:
# load the envelope-only model
rows = []
for subject in SUBS:
    for emotion in emotions:
        trf = eelbrain.load.unpickle(TRF_DIR / f'{subject}_{emotion}_phoneme_onset.pickle')
        rows.append([subject, emotion, trf.h, trf.proportion_explained, trf.r])

data_phoneme_onset = eelbrain.Dataset.from_caselist(
    ['subject', 'emotion', 'trf', 'det', 'cor'],
    rows
)

data_phoneme_onset['subject'] = Factor(data_phoneme_onset['subject'], random=True)
data_phoneme_onset['emotion'] = Factor(data_phoneme_onset['emotion'])

hap_phoneme_onset_ds = data_phoneme_onset.sub("emotion == 'hap'")
sad_phoneme_onset_ds = data_phoneme_onset.sub("emotion == 'sad'")

In [ ]:
print(hap_phoneme_onset_ds['cor'])
print(hap_phoneme_onset_ds['cor'].x)

In [ ]:
hap_test_phoneme_onset = eelbrain.testnd.TTestOneSample('cor', data = hap_phoneme_onset_ds, tail = 1, pmin = 0.05)
p = eelbrain.plot.Topomap(hap_test_phoneme_onset, clip = 'circle', w =2, vmax = -0.01, vmin = 0.01)
cb = p.plot_colorbar(width=0.1)
print(p.get_vlim())

In [ ]:
sad_test_phoneme_onset = eelbrain.testnd.TTestOneSample('cor', data = sad_phoneme_onset_ds, tail = 1, pmin = 0.05)
p = eelbrain.plot.Topomap(sad_test_phoneme_onset, clip = 'circle', w =2, vmax = -0.01, vmin = 0.01)

In [ ]:
res = eelbrain.testnd.TTestRelated(
    'cor',
    'emotion',
    'hap',
    'sad',
    match='subject',
    data=data_phoneme_onset,
    pmin=0.05,
)
p = eelbrain.plot.Topomap(res, clip='circle')

In [ ]:
print(res)

In [ ]:
print(hap_phoneme_onset_ds)

In [ ]:
hap_phoneme_trfs = hap_phoneme_onset_ds['trf'].mean(sensor = prp_sensors).smooth('time', window_samples=7, fix_edges=True)
sad_phoneme_trfs = sad_phoneme_onset_ds['trf'].mean(sensor = prp_sensors).smooth('time', window_samples=7, fix_edges=True)
test_phoneme_onset_hap_vs_sad = eelbrain.testnd.TTestRelated(
    hap_phoneme_trfs,
    sad_phoneme_trfs,
    pmin=0.05
)

p = eelbrain.plot.Array(
    test_phoneme_onset_hap_vs_sad,
    cmap='xpolar',
    axtitle=['hap', 'sad', 'hap vs sad']
)
for ax in p.axes:
    ax.tick_params(axis='y', labelsize=6) 
cb = p.plot_colorbar(width=0.1)

In [ ]:
print(test_phoneme_onset_hap_vs_sad)